In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
import numpy as np
import pandas as pd
import h5py
import os
from tqdm import tqdm 
from scipy import stats
from scipy.io import loadmat

/var/folders/km/7kbldtxn33136szl_s16122c0000gn/T/ipykernel_44990/1016450630.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


In [2]:
## helper functions for multi-day alignment

def standardize_alignment_columns(df):
    rename_map = {
        'Frame Number': 'FrameNumber',
        'Time Stamp (ms)': 'TimeStamp_ms_',
        'Buffer Index': 'BufferIndex'
    }
    return df.rename(columns=rename_map)


def nearest_timestamp_indices(reference_timestamps, query_timestamps):
    reference_timestamps = np.asarray(reference_timestamps)
    query_timestamps = np.asarray(query_timestamps)

    insert_positions = np.searchsorted(reference_timestamps, query_timestamps)
    insert_positions = np.clip(insert_positions, 0, len(reference_timestamps) - 1)
    previous_positions = np.clip(insert_positions - 1, 0, len(reference_timestamps) - 1)

    choose_previous = (
        np.abs(query_timestamps - reference_timestamps[previous_positions])
        <= np.abs(reference_timestamps[insert_positions] - query_timestamps)
    )

    return np.where(choose_previous, previous_positions, insert_positions)


def build_gap_index_ranges(videos_removed, video_length):
    return [
        ((video_start * video_length) - video_length, video_start * video_length)
        for video_start in videos_removed
    ]


def parse_removed_frame_ranges(removed_frame_ranges):
    idxs_list = []
    for frame_range in removed_frame_ranges:
        if isinstance(frame_range, (tuple, list)) and len(frame_range) == 2:
            start_idx, end_idx = int(frame_range[0]), int(frame_range[1])
        else:
            start_str, end_str = str(frame_range).split(':')
            start_idx = int(start_str)
            end_idx = int(end_str)

        if start_idx < 0 or end_idx < 0 or end_idx <= start_idx:
            raise ValueError(
                f"Invalid removed frame range '{frame_range}'. Expected 0 <= start < end."
            )

        idxs_list.append((start_idx, end_idx))

    return idxs_list


def build_day_gap_index_ranges(day_config):
    idxs_list = build_gap_index_ranges(
        day_config.get('videos_removed', []),
        day_config.get('video_length', 1000),
    )
    idxs_list.extend(parse_removed_frame_ranges(day_config.get('removed_frame_ranges', [])))
    return idxs_list


def drop_gap_frames(df, idxs_list):
    if not idxs_list:
        return df

    to_drop = []
    for start, end in idxs_list:
        to_drop.extend(range(start, end))

    if any(idx >= len(df) for idx in to_drop):
        raise ValueError(
            f'At least one removed frame index is outside the aligned data length ({len(df)}).'
        )

    labels_to_drop = df.index[to_drop]
    return df.drop(labels=labels_to_drop)


def alignMiniscopeBehavCamTimestamps(
    sessionPath,
    behavTimestampFile,
    miniscopeTimestampFile,
    day_settings,
):
    session_dir = os.path.dirname(sessionPath)

    ezTrackOutput = standardize_alignment_columns(pd.read_csv(sessionPath))
    timestampfile = standardize_alignment_columns(pd.read_csv(os.path.join(session_dir, behavTimestampFile)))
    miniscope_timestampfile = standardize_alignment_columns(pd.read_csv(os.path.join(session_dir, miniscopeTimestampFile)))

    required_timestamp_cols = {'FrameNumber', 'TimeStamp_ms_', 'Day'}
    required_tracking_cols = {'Frame', 'X', 'Y', 'Distance_px', 'Day'}

    if not required_timestamp_cols.issubset(timestampfile.columns):
        raise ValueError(f'Behavior timestamp file is missing required columns: {required_timestamp_cols}')
    if not required_timestamp_cols.issubset(miniscope_timestampfile.columns):
        raise ValueError(f'Miniscope timestamp file is missing required columns: {required_timestamp_cols}')
    if not required_tracking_cols.issubset(ezTrackOutput.columns):
        raise ValueError(f'Behavior tracking file is missing required columns: {required_tracking_cols}')

    aligned_days = []
    common_days = sorted(
        set(ezTrackOutput['Day']).intersection(timestampfile['Day']).intersection(miniscope_timestampfile['Day'])
    )

    if not common_days:
        raise ValueError('No overlapping days were found across the calcium, behavior, and timestamp files.')

    for day in common_days:
        if day not in day_settings:
            raise ValueError(f'Missing settings for Day {day} in DAY_SETTINGS.')

        day_config = day_settings[day]
        behav_day = timestampfile.loc[timestampfile['Day'] == day].reset_index(drop=True)
        miniscope_day = miniscope_timestampfile.loc[miniscope_timestampfile['Day'] == day].reset_index(drop=True).copy()
        tracking_day = ezTrackOutput.loc[ezTrackOutput['Day'] == day].reset_index(drop=True)

        if len(behav_day) != len(tracking_day):
            raise ValueError(
                f'Day {day}: behavior timestamps ({len(behav_day)}) and tracking rows ({len(tracking_day)}) do not match.'
            )

        closest_behav_indices = nearest_timestamp_indices(
            behav_day['TimeStamp_ms_'].to_numpy(),
            miniscope_day['TimeStamp_ms_'].to_numpy()
        )

        miniscope_day['closestBehavCamFrameIdx_within_day'] = closest_behav_indices
        miniscope_day['closestBehavCamFrameIdx'] = behav_day.loc[closest_behav_indices, 'FrameNumber'].to_numpy()
        miniscope_day['X_coor'] = tracking_day.loc[closest_behav_indices, 'X'].to_numpy()
        miniscope_day['Y_coor'] = tracking_day.loc[closest_behav_indices, 'Y'].to_numpy()
        miniscope_day['Distance_px'] = tracking_day.loc[closest_behav_indices, 'Distance_px'].to_numpy()

        miniscope_day = drop_gap_frames(miniscope_day, build_day_gap_index_ranges(day_config))
        aligned_days.append(miniscope_day)

    return pd.concat(aligned_days, ignore_index=True)


def summarize_day_settings(day_settings):
    for day in sorted(day_settings):
        config = day_settings[day]
        removed_ranges = build_day_gap_index_ranges(config)
        print(
            f"Day {day}: behavior {config['behavior_frame_rate']} Hz, "
            f"miniscope {config['miniscope_frame_rate']} Hz, "
            f"removed ranges {removed_ranges}"
        )

In [3]:
## load and do some preprocessing on the CNMFE traces 
def normalize(trace, percentile=True):
    """ Normalize a fluorescence trace by its max or its 99th percentile. """
    trace = trace - np.min(trace)
    if np.percentile(trace, 99) > 0:
        if percentile:
            trace = trace / np.percentile(trace, 99)
        else:
            trace = trace / np.max(trace)
    return trace

def load_and_filter_traces(extract_mat_path: str,
                           labels_mat_path: str,
                           label_key: str = 'labels_ex'
                          ) -> pd.DataFrame:
    """
    Load temporal_weights from extract_mat_path (v7.3) and one of the
    labels_* arrays from labels_mat_path, keep only columns where
    labels == 1, and return as a pandas DataFrame.

    Parameters
    ----------
    extract_mat_path : str
        Path to the v7.3 .mat file containing an 'output/temporal_weights' dataset.
    labels_mat_path : str
        Path to the .mat file (v7.3 or earlier) containing a 'labels' group/struct.
    label_key : str, optional
        Which labels field to use: one of 'labels_ex', 'labels_ml', or
        'labels_overall'. Default is 'labels_ex'.

    Returns
    -------
    pd.DataFrame
        Rows = frames, columns = kept cells (named 'cell_<original_index>').
    """
    # validate choice
    allowed = ('labels_ex','labels_ml','labels_overall')
    if label_key not in allowed:
        raise ValueError(f"label_key must be one of {allowed}, got '{label_key}'")

    # 1) load & transpose temporal_weights → shape (nFrames, nCells)
    with h5py.File(extract_mat_path, 'r') as f:
        tw = f['output']['temporal_weights'][()]  # often (nCells, nFrames)
    tw = np.asarray(tw).T

    # 2) try to load chosen labels via HDF5; if that fails, fall back to loadmat
    try:
        with h5py.File(labels_mat_path, 'r') as f:
            hl = f['labels'][label_key][()]
    except OSError:
        mat = loadmat(labels_mat_path,
                      struct_as_record=False,
                      squeeze_me=True)
        lbl = mat['labels']  # either a dict or a mat_struct
        # pull out the right attribute/key
        if isinstance(lbl, dict):
            hl = lbl[label_key]
        else:
            hl = getattr(lbl, label_key)

    # 3) shape‐check & squeeze
    hl = np.asarray(hl).squeeze()
    if tw.shape[1] != hl.size:
        raise ValueError(
            f"dimension mismatch: temporal_weights is {tw.shape}, "
            f"labels array '{label_key}' has length {hl.size}"
        )

    # 4) filter & build DataFrame
    mask    = (hl == 1)
    kept_i  = np.nonzero(mask)[0]
    filtered = tw[:, mask]
    cols    = [f'cell_{i}' for i in kept_i]

    return (pd.DataFrame(filtered, columns=cols), hl)

def print_h5_tree(name, obj):
    """
    Callback for h5py.File.visititems.
    Prints group/dataset name and, for datasets, its shape and dtype.
    """
    if isinstance(obj, h5py.Group):
        print(f"Group:   {name}/")
    elif isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}  — shape={obj.shape}, dtype={obj.dtype}")

    
def zScoreTraces(dirPath, CNMFE_real_cells):
    
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    C_normalized = CNMFE_real_cells.apply(lambda col: normalize(col), axis=0)
    C_z_scored = CNMFE_real_cells.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(CNMFE_real_cells)-1)*(1/20), len(CNMFE_real_cells)), unit='s'), drop=True)
    C_normalized_z_scored = C_normalized.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(C_normalized)-1)*(1/20), len(C_normalized)), unit='s'), drop=True)

    ##load spatial components by session
    # for v4 dimensions are 600x600 pixels
 
    C_normalized_z_scored.to_csv(dirPath+'_C_traces_filtered_origHz.csv')
    
    print('finished, saved:')
    print(dirPath+'_C_traces_filtered_origHz.csv')
    
    return(C_normalized_z_scored)

def zScoreTraces_withGappedTime(dirPath, CNMFE_real_cells, miniscopeFramesPerSecond, idxsList):
    # Convert all columns to numeric (as before)
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    # 1) Build a list of *all* original frame‐numbers (0…N−1),
    #    then remove the “excised” ranges to get `kept_frames`.
    #    N = M + total_removed
    removed_indices = []
    for start, end in idxsList:
        # remove [start, end) from original
        removed_indices.extend(range(start, end))
    removed_set = set(removed_indices)
    # total frames originally = len(kept) + len(removed)
    total_original_frames = len(CNMFE_real_cells) + len(removed_set)
    # generate all original indices 0…N−1
    all_indices = np.arange(total_original_frames)
    # keep only those not in removed_set, in sorted order
    kept_frames = np.array([i for i in all_indices if i not in removed_set])
    # sanity‐check: kept_frames.size == len(CNMFE_real_cells)
    if kept_frames.shape[0] != len(CNMFE_real_cells):
        raise ValueError(
            "Number of kept frames (%d) != number of rows in CNMFE_real_cells (%d)"
            % (kept_frames.shape[0], len(CNMFE_real_cells))
        )
    # 2) Create a "gapped" timedelta index from those kept_frames:
    #    time (in seconds) = frame_number * (1 / samplingRate)
    times_sec = kept_frames * (1.0 / miniscopeFramesPerSecond)
    time_index = pd.to_timedelta(times_sec, unit='s')
    # 3) Now normalize & z‐score (unchanged from before), but *do not* set the index yet:
    C_normalized = CNMFE_real_cells.apply(lambda col: (col - col.min()) / (col.max() - col.min()), axis=0)
    C_z_scored   = CNMFE_real_cells.apply(stats.zscore, axis=0)
    C_norm_z     = C_normalized.apply(stats.zscore, axis=0)
    # 4) Finally, assign our custom `time_index` to both:
    C_z_scored.index   = time_index
    C_norm_z.index     = time_index
    # 5) (Optional) save to CSV
    C_norm_z.to_csv(dirPath + '_C_traces_filtered_origHz.csv')
    print("Finished, saved to:", dirPath + '_C_traces_filtered_origHz.csv')
    return C_norm_z, C_z_scored

In [4]:
# edit only this cell for a new recording

dirPath = os.getcwd()
trace_csv_name = '_C_traces_filtered_origHz.csv'
session_csv_name = 'Mouse20_combined_1_6_12.csv'
behav_timestamp_csv_name = 'timeStamps_Mouse20_Combined_1_6_12.csv'
miniscope_timestamp_csv_name = 'timeStampsMiniscope_Mouse20_Combined_1_6_12.csv'
aligned_output_name = 'Mouse20_combined_1_6_12_cellTracesAlignedToTracking.csv'

DAY_SETTINGS = {
    1: {
        'behavior_frame_rate': 15,
        'miniscope_frame_rate': 25,
        'videos_removed': [],
        'video_length': 1000,
        'removed_frame_ranges': [],
    },
    2: {
        'behavior_frame_rate': 15,
        'miniscope_frame_rate': 20,
        'videos_removed': [],
        'video_length': 1000,
        'removed_frame_ranges': [],
    },
    3: {
        'behavior_frame_rate': 15,
        'miniscope_frame_rate': 30,
        'videos_removed': [],
        'video_length': 1000,
        'removed_frame_ranges': [],
    },
}

trace_csv_path = os.path.join(dirPath, trace_csv_name)
sessionPath = os.path.join(dirPath, session_csv_name)
behavTimestampPath = os.path.join(dirPath, behav_timestamp_csv_name)
miniscopeTimestampPath = os.path.join(dirPath, miniscope_timestamp_csv_name)
aligned_output_path = os.path.join(dirPath, aligned_output_name)

GCAMP_traces_ZscoreNormalized_Gapped = pd.read_csv(trace_csv_path, index_col=0)
GCAMP_traces_ZscoreNormalized_Gapped.index = pd.to_timedelta(GCAMP_traces_ZscoreNormalized_Gapped.index)

summarize_day_settings(DAY_SETTINGS)
print(f'Loaded {len(GCAMP_traces_ZscoreNormalized_Gapped)} calcium frames from {trace_csv_path}')

Day 1: behavior 15 Hz, miniscope 25 Hz, removed ranges []
Day 2: behavior 15 Hz, miniscope 20 Hz, removed ranges []
Day 3: behavior 15 Hz, miniscope 30 Hz, removed ranges []
Loaded 152681 calcium frames from /Users/charlottecastillon/Desktop/untitled folder 2/_C_traces_filtered_origHz.csv


In [5]:
# if you loaded labels from the original .mat files above, this compares human vs model labels
if 'labels_human' in globals() and 'labels_model' in globals():
    diff_inds = np.where(labels_human != labels_model)[0]
    diff_inds
else:
    print('Skipping label comparison because the notebook is using the preprocessed calcium CSV.')

Skipping label comparison because the notebook is using the preprocessed calcium CSV.


In [6]:
# align behavior to the miniscope timestamps, using the settings for each day
behavCamDataAligned_clean = alignMiniscopeBehavCamTimestamps(
    sessionPath=sessionPath,
    behavTimestampFile=behav_timestamp_csv_name,
    miniscopeTimestampFile=miniscope_timestamp_csv_name,
    day_settings=DAY_SETTINGS,
)

print(f'Aligned {len(behavCamDataAligned_clean)} miniscope frames across days {sorted(behavCamDataAligned_clean["Day"].unique())}')

Aligned 152681 miniscope frames across days [np.int64(1), np.int64(2), np.int64(3)]


In [7]:
# merge the aligned behavior data with the calcium traces
if len(GCAMP_traces_ZscoreNormalized_Gapped) != len(behavCamDataAligned_clean):
    raise ValueError(
        f'Calcium trace rows ({len(GCAMP_traces_ZscoreNormalized_Gapped)}) do not match aligned behavior rows ({len(behavCamDataAligned_clean)}).'
    )

# keep the calcium trace time index in the final table
behavCamDataAligned_clean.index = GCAMP_traces_ZscoreNormalized_Gapped.index

CNMFE_aligned = pd.concat(
    [GCAMP_traces_ZscoreNormalized_Gapped, behavCamDataAligned_clean],
    axis=1
)

CNMFE_aligned.to_csv(aligned_output_path)
print(f'Saved aligned calcium + behavior data to: {aligned_output_path}')

Saved aligned calcium + behavior data to: /Users/charlottecastillon/Desktop/untitled folder 2/Mouse20_combined_1_6_12_cellTracesAlignedToTracking.csv


In [8]:
CNMFE_aligned

,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,cell_11,...,cell_166,FrameNumber,TimeStamp_ms_,BufferIndex,Day,closestBehavCamFrameIdx_within_day,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px
0 days 00:00:00,-1.301506,2.530391,-1.373509,0.605313,1.580661,-1.327305,-1.332990,-1.017508,-0.096174,-0.454462,...,-0.645213,0,-22,0,1,0,0,34.512974,51.031936,0.000000
0 days 00:00:00.040000,-0.930685,0.902377,-0.050944,-0.325893,1.323257,-1.317027,-0.300860,-0.970136,-1.409240,1.474604,...,-0.162860,1,21,0,1,0,0,34.512974,51.031936,0.000000
0 days 00:00:00.080000,-0.629057,-0.772504,-0.178821,0.663097,0.022159,1.668466,-0.728895,1.198399,-0.758549,1.106448,...,-0.436298,2,58,0,1,1,1,34.328279,51.151902,0.220237
0 days 00:00:00.120000,1.852487,-1.165005,1.490117,-0.859983,-1.537425,-0.621496,-0.150396,-0.282884,-0.582781,2.036563,...,-1.308009,3,99,0,1,1,1,34.328279,51.151902,0.220237
0 days 00:00:00.160000,1.400981,-0.390942,0.720933,-1.172837,-1.211102,-0.895404,-0.994599,-0.112402,-0.309144,0.630488,...,-0.279713,4,145,0,1,2,2,34.489830,51.235569,0.181931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0 days 01:41:47.040000,1.297600,-0.032896,-0.042770,-0.659493,0.439316,0.519942,1.113716,0.116817,-1.177430,-0.286679,...,1.182860,152676,5445165,0,3,26072,78061,589.645021,21.728662,1.042997
0 days 01:41:47.080000,-0.048756,-0.621892,-0.741475,0.469569,-0.541063,1.011584,0.618741,1.157346,0.588595,-0.568211,...,-0.188854,152677,5445199,0,3,26072,78061,589.645021,21.728662,1.042997
0 days 01:41:47.120000,1.034348,-1.249222,2.122364,-0.316486,-0.225361,0.083942,0.196229,0.282131,0.166663,0.825875,...,-0.705691,152678,5445233,0,3,26072,78061,589.645021,21.728662,1.042997
0 days 01:41:47.160000,0.019489,-0.343812,-0.843587,0.678057,-0.770125,-0.820432,-0.580946,1.434697,0.650234,-0.801168,...,-1.040083,152679,5445270,0,3,26072,78061,589.645021,21.728662,1.042997
